# Fase 2 - Entendimento dos dados

Este notebook tem como objetivo realizar a primeira análise estrutural dos dados utilizados no projeto de predição de atraso nas entregas da empresa NexaMarket.

Nesa etapa serão avaliados:

- arquivos disponíveis;
- dimensões das tabelas;
- tipos de dados;
- valores ausentes;
- chaves de relacionamento;
- possíveis problemas de qualidade;
- variáveis permitidas e proibidas para modelagem.

Ainda não será realizada modelagem preditiva nessa fase.

### Importações

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

### Configurações gerais

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', None)


### Paths

In [4]:
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'data' / 'raw').is_dir()
)
RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'

### Listagem de arquivos

In [5]:
csv_files = sorted(RAW_DATA_PATH.glob('*.csv'))

for file in csv_files:
    print(file)

/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-delivery-risk/data/raw/olist_customers_dataset.csv
/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-delivery-risk/data/raw/olist_geolocation_dataset.csv
/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-delivery-risk/data/raw/olist_order_items_dataset.csv
/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-delivery-risk/data/raw/olist_order_payments_dataset.csv
/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-delivery-risk/data/raw/olist_order_reviews_dataset.csv
/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-delivery-risk/data/raw/olist_orders_dataset.csv
/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-delivery-risk/data/raw/olist_products_dataset.csv
/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-delivery-risk/data/raw/olist_sellers_dataset.csv
/Users/diegosantos/Documents/Cérebro/02-Areas/Pessoal/ecommerce-

### Carregamento das tabelas

In [6]:
(
    clientes,
    coordenadas,
    itens_pedido,
    pagamentos,
    avaliacoes,
    pedidos,
    produtos,
    vendedores,
    produto_categoria,
) = map(pd.read_csv, csv_files)

In [28]:
print(f'Total de arquivos carregados: {len(csv_files)}')

Total de arquivos carregados: 9


### Dicionário de tabelas

In [7]:
tabelas = {
    'clientes': clientes,
    'coordenadas': coordenadas,
    'itens_pedido': itens_pedido,
    'pagamentos': pagamentos,
    'avaliacoes': avaliacoes,
    'pedidos': pedidos,
    'produtos': produtos,
    'vendedores': vendedores,
    'produto_categoria': produto_categoria,
}

### Dimensões das tabelas

In [8]:
summary = []

for nome, df in tabelas.items():
    summary.append({
        'tabela': nome,
        'linhas': df.shape[0],
        'colunas': df.shape[1],
        'linhas_duplicadas': df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)
summary_df

,tabela,linhas,colunas,linhas_duplicadas
0,clientes,99441,5,0
1,coordenadas,1000163,5,261831
2,itens_pedido,112650,7,0
3,pagamentos,103886,5,0
4,avaliacoes,99224,7,0
5,pedidos,99441,8,0
6,produtos,32951,9,0
7,vendedores,3095,4,0
8,produto_categoria,71,2,0


### Visualização inicial das colunas

In [9]:
for nome, df in tabelas.items():
    print(f'\n Tabela: {nome}')
    print(df.columns.tolist())


 Tabela: clientes
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

 Tabela: coordenadas
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

 Tabela: itens_pedido
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

 Tabela: pagamentos
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

 Tabela: avaliacoes
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

 Tabela: pedidos
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

 Tabela: produtos
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty'

### Tipos de dados

In [10]:
for nome, ds in tabelas.items():
    print(f"\n{'=' * 80}")
    print(f"Tabela: {nome}")
    print(df.info())


Tabela: clientes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB
None

Tabela: coordenadas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB
None

Tabela: itens_pedido
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------ 

### Valores ausentes

In [11]:
def missing_values_summary(tabelas):
    missing_summary = []

    for nome, ds in tabelas.items():
        missing = ds.isna().sum()
        missinc_percent = (missing / len(ds)) * 100

        temp = pd.DataFrame({
            'tabela': nome,
            'coluna': missing.index,
            'ausentes': missing.values,
            'percentual': missinc_percent.values
            })

        # filtra colunas com valores ausentes
        temp = temp[temp['ausentes'] > 0]
        missing_summary.append(temp)

    missing_df = pd.concat(missing_summary, ignore_index=True)
    print(missing_df.sort_values(by=['tabela', 'percentual'], ascending=[True, False]))

In [12]:
missing_values_summary(tabelas)

        tabela                         coluna  ausentes  percentual
0   avaliacoes           review_comment_title     87656       88.34
1   avaliacoes         review_comment_message     58247       58.70
4      pedidos  order_delivered_customer_date      2965        2.98
3      pedidos   order_delivered_carrier_date      1783        1.79
2      pedidos              order_approved_at       160        0.16
5     produtos          product_category_name       610        1.85
6     produtos            product_name_lenght       610        1.85
7     produtos     product_description_lenght       610        1.85
8     produtos             product_photos_qty       610        1.85
9     produtos               product_weight_g         2        0.01
10    produtos              product_length_cm         2        0.01
11    produtos              product_height_cm         2        0.01
12    produtos               product_width_cm         2        0.01


### Verificação das principais chaves

In [13]:
keys_checks = {
    'pedidos_pedidos_id_is_unique' : pedidos['order_id'].is_unique,
    'clientes_cliente_id_is_unique' : clientes['customer_id'].is_unique,
    'produtos_produto_id_is_unique' : produtos['product_id'].is_unique,
    'vendedores_vendedor_id_is_unique' : vendedores['seller_id'].is_unique
}

for nome, is_unique in keys_checks.items():
    print(f'{nome}: {is_unique}')
    

pedidos_pedidos_id_is_unique: True
clientes_cliente_id_is_unique: True
produtos_produto_id_is_unique: True
vendedores_vendedor_id_is_unique: True


### Cardinalidade das relações

In [18]:
relationship_checks = {
    'pedidos_in_item_pedido' : pedidos['order_id'].isin(itens_pedido['order_id']).mean(),
    'pedidos_in_pagamentos' : pedidos['order_id'].isin(pagamentos['order_id']).mean(),
    'pedidos_in_avaliacoes' : pedidos['order_id'].isin(avaliacoes['order_id']).mean(),
    'clientes_in_pedidos' : clientes['customer_id'].isin(pedidos['customer_id']).mean(),
    'produtos_in_itens_pedido' : produtos['product_id'].isin(itens_pedido['product_id']).mean(),
    'vendedores_in_itens_pedido' : vendedores['seller_id'].isin(itens_pedido['seller_id']).mean()
}

for nome, is_in in relationship_checks.items():
    print(f'{nome}: {is_in}')
    
### Verificação de valores fora do padrão




pedidos_in_item_pedido: 0.9922064339658692
pedidos_in_pagamentos: 0.9999899437857624
pedidos_in_avaliacoes: 0.9922768274655324
clientes_in_pedidos: 1.0
produtos_in_itens_pedido: 1.0
vendedores_in_itens_pedido: 1.0


### Conversão inicial das datas

In [19]:
date_columns_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns_orders:
    pedidos[col] = pd.to_datetime(pedidos[col], errors='coerce')

pedidos.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


### Período dos pedidos

In [20]:
pedidos['order_purchase_timestamp'].agg(['min', 'max'])

min   2016-09-04 21:15:19
max   2018-10-17 17:30:18
Name: order_purchase_timestamp, dtype: datetime64[ns]

### Distribuição dos status dos pedidos

In [22]:
pedidos['order_status'].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### Primeira criação da variável-alvo

In [24]:
pedidos_entregues = pedidos[
    (pedidos['order_status'] == 'delivered') &
    (pedidos['order_delivered_customer_date'].notna()) &
    (pedidos['order_estimated_delivery_date'].notna())
    ].copy()

pedidos_entregues['atrasou_entrega'] = (
    pedidos_entregues['order_delivered_customer_date'] >
    pedidos_entregues['order_estimated_delivery_date']
).astype(int)
 
pedidos_entregues['atrasou_entrega'].value_counts(normalize=True)

atrasou_entrega
0   0.92
1   0.08
Name: proportion, dtype: float64

### Taxa de atraso

In [26]:
taxa_atraso = pedidos_entregues['atrasou_entrega'].mean()

print(f'Taxa de atraso: {taxa_atraso:.2%}')
print(f'Pedidos entregues analisados: {pedidos_entregues.shape[0]}')

Taxa de atraso: 8.11%
Pedidos entregues analisados: 96470
